In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.8.0+cpu
False


In [5]:
import torch
print(torch.__version__)          # kalau ada "+cpu" di belakang -> itu CPU-only build

2.8.0+cpu


In [6]:
# ============================================================
# NOTEBOOK BARU: Modeling ConvLSTM — load dari bundle offline
# Gak butuh ee.Initialize() sama sekali, semua data udah lokal
# ============================================================
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

BUNDLE_DIR = "../data/tensors/offline_adaptive"   # hasil merge_and_save kemarin
LOOKBACK   = 3
TRAIN_UNTIL = 2022

def load_offline(bundle_dir):
    with open(os.path.join(bundle_dir, 'manifest.json')) as f:
        manifest = json.load(f)
    masks = {}
    for fn in os.listdir(bundle_dir):
        if not fn.endswith('_masks.npz'):
            continue
        nama = fn.replace('_masks.npz', '')
        z = np.load(os.path.join(bundle_dir, fn))
        masks[nama] = {(int(k.split('_')[0]), k.split('_')[1]): z[k] for k in z.files}
    print(f"Loaded {len(masks)} AOI dari {bundle_dir} (dibuat {manifest['created_utc'][:10]})")
    return masks, manifest

masks, manifest = load_offline(BUNDLE_DIR)
print("AOI:", list(masks.keys()))

device: cpu
Loaded 9 AOI dari ../data/tensors/offline_adaptive (dibuat 2026-07-16)
AOI: ['Titik_02', 'Titik_04', 'Titik_05', 'Titik_06', 'Titik_07', 'Titik_09', 'Titik_10', 'Titik_11', 'Titik_19']


In [7]:
SEASON_ORDER = ['S1', 'S2', 'S3']

def seq_index(yr, s):
    return yr * 3 + SEASON_ORDER.index(s)

def build_tensor(masks, lookback=LOOKBACK):
    X_list, y_list, meta = [], [], []
    for nama, frames in masks.items():
        keys = sorted(frames.keys(), key=lambda k: seq_index(*k))
        for i in range(len(keys) - lookback):
            win, target = keys[i:i+lookback], keys[i+lookback]
            idx = [seq_index(*k) for k in win + [target]]
            if idx != list(range(idx[0], idx[0] + lookback + 1)):
                continue   # window nyebrang gap -> skip
            X_list.append(np.stack([frames[k][np.newaxis] for k in win]))
            y_list.append(frames[target][np.newaxis])
            meta.append({'aoi': nama, 'input': win, 'target': target,
                         'target_year': target[0]})
    X = np.stack(X_list)   # (N, LOOKBACK, 1, 128, 128)
    y = np.stack(y_list)   # (N, 1, 128, 128)
    print(f"Tensor: X{X.shape} y{y.shape} | {len(meta)} sample dari {len(masks)} AOI")
    return X, y, meta

X, y, meta = build_tensor(masks)

def temporal_split(X, y, meta, train_until=TRAIN_UNTIL):
    tr = [i for i, m in enumerate(meta) if m['target_year'] <= train_until]
    te = [i for i, m in enumerate(meta) if m['target_year'] >  train_until]
    print(f"train: {len(tr)} | test: {len(te)}")
    return (torch.from_numpy(X[tr]).float(), torch.from_numpy(y[tr]).float(),
            torch.from_numpy(X[te]).float(), torch.from_numpy(y[te]).float(),
            [meta[i] for i in tr], [meta[i] for i in te])

X_train, y_train, X_test, y_test, meta_train, meta_test = temporal_split(X, y, meta)
print("X_train:", X_train.shape, "| y_train:", y_train.shape)

Tensor: X(176, 3, 1, 128, 128) y(176, 1, 128, 128) | 176 sample dari 9 AOI
train: 98 | test: 78
X_train: torch.Size([98, 3, 1, 128, 128]) | y_train: torch.Size([98, 1, 128, 128])


In [8]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        # 4 gate (i, f, o, g) sekaligus dalam 1 conv, dipecah setelahnya
        self.conv = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch,
                              kernel_size, padding=pad)
        self.hidden_ch = hidden_ch

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)          # (B, in+hidden, H, W)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch, H, W, device):
        return (torch.zeros(batch, self.hidden_ch, H, W, device=device),
                torch.zeros(batch, self.hidden_ch, H, W, device=device))


class ShorelineConvLSTM(nn.Module):
    """Input: (B, T, C=1, H, W) mask biner sequence
    Output: (B, 1, H, W) probabilitas water (sigmoid), BUKAN mask langsung.
    Threshold buat binarize ditentukan belakangan lewat sweep di validation
    (sesuai keputusan locked: Dice/BCE loss, decision threshold via sweep)."""

    def __init__(self, in_ch=1, hidden_ch=16, kernel_size=3, n_layers=1):
        super().__init__()
        layers = []
        for i in range(n_layers):
            layers.append(ConvLSTMCell(in_ch if i == 0 else hidden_ch,
                                       hidden_ch, kernel_size))
        self.cells = nn.ModuleList(layers)
        self.head = nn.Conv2d(hidden_ch, 1, kernel_size=1)   # -> 1 channel logit

    def forward(self, x):
        B, T, C, H, W = x.shape
        h = [None] * len(self.cells)
        c = [None] * len(self.cells)
        for l, cell in enumerate(self.cells):
            h[l], c[l] = cell.init_hidden(B, H, W, x.device)

        for t in range(T):
            inp = x[:, t]
            for l, cell in enumerate(self.cells):
                h[l], c[l] = cell(inp, h[l], c[l])
                inp = h[l]

        logit = self.head(h[-1])           # (B, 1, H, W)
        return logit                        # BCEWithLogits handle sigmoid-nya

In [9]:
def dice_score(pred, target, eps=1e-6):
    pred, target = pred.flatten(), target.flatten()
    inter = (pred * target).sum()
    return (2*inter + eps) / (pred.sum() + target.sum() + eps)

# baseline: tebak "frame depan = frame input terakhir"
persist_pred = X_test[:, -1]   # (N, 1, 128, 128), frame terakhir di window
dice_baseline = dice_score(persist_pred, y_test).item()
print(f"Persistence baseline Dice (test set): {dice_baseline:.4f}")
print("^ ini tembok yang HARUS dilewati ConvLSTM biar berarti apa-apa")

Persistence baseline Dice (test set): 0.9434
^ ini tembok yang HARUS dilewati ConvLSTM biar berarti apa-apa


In [10]:
class DiceBCELoss(nn.Module):
    """Kombinasi Dice + BCE, sesuai keputusan: class balance 1:1.
    BCE stabil buat gradient awal, Dice fokus ke overlap region
    (penting krn shoreline itu boundary tipis, class bisa imbalanced
    kalau AOI didominasi laut/darat)."""
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight

    def dice_loss(self, logit, target, eps=1e-6):
        prob = torch.sigmoid(logit)
        prob, target = prob.flatten(1), target.flatten(1)
        inter = (prob * target).sum(1)
        dice = (2*inter + eps) / (prob.sum(1) + target.sum(1) + eps)
        return 1 - dice.mean()

    def forward(self, logit, target):
        return (self.bce_weight * self.bce(logit, target)
                + (1 - self.bce_weight) * self.dice_loss(logit, target))

criterion = DiceBCELoss(bce_weight=0.5)

In [11]:
# def overfit_one_sample(model, x, y, steps=300, lr=1e-3, log_every=50):
#     """Model sehat HARUS bisa hafal 1 sample sampai loss mendekati 0.
#     Kalau gagal: bug di arsitektur/loss/data pipeline, bukan hyperparameter."""
#     model.train()
#     opt = torch.optim.Adam(model.parameters(), lr=lr)
#     losses = []
#     for step in range(steps):
#         opt.zero_grad()
#         logit = model(x)
#         loss = criterion(logit, y)
#         loss.backward()
#         opt.step()
#         losses.append(loss.item())
#         if step % log_every == 0 or step == steps - 1:
#             with torch.no_grad():
#                 pred = (torch.sigmoid(logit) > 0.5).float()
#                 d = dice_score(pred, y).item()
#             print(f"step {step:4d} | loss={loss.item():.4f} | dice={d:.4f}")
#     return losses

# model = ShorelineConvLSTM(in_ch=1, hidden_ch=16, n_layers=1).to(device)
# x1, y1 = X_train[:1].to(device), y_train[:1].to(device)
# losses = overfit_one_sample(model, x1, y1)

# plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss')
# plt.title('Overfit 1 sample — harus turun mendekati 0')
# plt.show()

In [ ]:
import logging
import csv
import os
from datetime import datetime

# ---------- setup logger ----------
LOG_DIR = "../logs"
os.makedirs(LOG_DIR, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = os.path.join(LOG_DIR, f"train_{run_id}.log")

logger = logging.getLogger("shoreline_train")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # biar gak dobel handler kalau cell di-run ulang

fh = logging.FileHandler(log_path)
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S"))
logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(sh)

logger.info(f"=== Run {run_id} start ===")


def train_model(model, X_train, y_train, X_test, y_test,
                epochs=150, lr=1e-3, batch_size=8, log_every=10,
                csv_path=None, use_tensorboard=False, tb_logdir=None):
    """Training loop dengan logging file + CSV metrik per-epoch.
    csv_path: kalau None, auto ../logs/metrics_{run_id}.csv
    use_tensorboard: kalau True, log juga ke TensorBoard (opsional, gak wajib)
    """
    csv_path = csv_path or os.path.join(LOG_DIR, f"metrics_{run_id}.csv")
    writer_tb = None
    if use_tensorboard:
        from torch.utils.tensorboard import SummaryWriter
        tb_logdir = tb_logdir or os.path.join(LOG_DIR, "tb", run_id)
        writer_tb = SummaryWriter(tb_logdir)
        logger.info(f"TensorBoard aktif: tensorboard --logdir {os.path.join(LOG_DIR, 'tb')}")

    # tulis header CSV sekali di awal
    with open(csv_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "train_dice", "test_dice"])

    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(X_train)
    history = {'train_loss': [], 'train_dice': [], 'test_dice': []}

    logger.info(f"Mulai training: {n} sample train, {len(X_test)} test, "
               f"epochs={epochs}, batch_size={batch_size}, lr={lr}, device={device}")

    try:
        for epoch in range(epochs):
            perm = torch.randperm(n)
            epoch_loss = 0.0
            for i in range(0, n, batch_size):
                idx = perm[i:i+batch_size]
                xb, yb = X_train[idx].to(device), y_train[idx].to(device)
                opt.zero_grad()
                logit = model(xb)
                loss = criterion(logit, yb)
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(idx)
                del logit, loss   # cleanup graf tiap batch
            epoch_loss /= n

            model.eval()
            with torch.no_grad():
                tr_logit = model(X_train.to(device))
                tr_dice = dice_score((torch.sigmoid(tr_logit) > 0.5).float(),
                                     y_train.to(device)).item()
                te_logit = model(X_test.to(device))
                te_dice = dice_score((torch.sigmoid(te_logit) > 0.5).float(),
                                     y_test.to(device)).item()
                del tr_logit, te_logit   # cleanup — kecurigaan leak kemarin
            model.train()

            history['train_loss'].append(epoch_loss)
            history['train_dice'].append(tr_dice)
            history['test_dice'].append(te_dice)

            # simpan tiap epoch -- kalau crash, histori sampai sini tetap ada
            with open(csv_path, "a", newline="") as f:
                csv.writer(f).writerow([epoch, epoch_loss, tr_dice, te_dice])

            if writer_tb:
                writer_tb.add_scalar("loss/train", epoch_loss, epoch)
                writer_tb.add_scalar("dice/train", tr_dice, epoch)
                writer_tb.add_scalar("dice/test", te_dice, epoch)

            if epoch % log_every == 0 or epoch == epochs - 1:
                logger.info(f"epoch {epoch:3d} | loss={epoch_loss:.4f} | "
                           f"train_dice={tr_dice:.4f} | test_dice={te_dice:.4f}")

    except Exception as e:
        logger.error(f"Training berhenti di epoch {epoch}: {type(e).__name__}: {e}")
        logger.info(f"Histori sampai epoch {epoch-1} tersimpan di {csv_path}")
        raise
    finally:
        if writer_tb:
            writer_tb.close()

    logger.info(f"Training selesai. Metrik: {csv_path} | Log: {log_path}")
    return history




=== Run 20260716_222524 start ===


: 

In [ ]:
model = ShorelineConvLSTM(in_ch=1, hidden_ch=16, n_layers=1).to(device)
history = train_model(model, X_train, y_train, X_test, y_test,
                      epochs=150, use_tensorboard=False)   # ganti True kapan pun mau

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss']); ax[0].set_title('Training loss')
ax[1].plot(history['train_dice'], label='train')
ax[1].plot(history['test_dice'], label='test')
ax[1].axhline(dice_baseline, color='r', ls='--', label=f'persistence baseline ({dice_baseline:.3f})')
ax[1].legend(); ax[1].set_title('Dice: train vs test vs baseline')
plt.tight_layout(); plt.show()

logger.info(f"Final test Dice: {history['test_dice'][-1]:.4f} | "
           f"baseline: {dice_baseline:.4f} | "
           f"gap: {history['train_dice'][-1] - history['test_dice'][-1]:.4f}")

Mulai training: 98 sample train, 78 test, epochs=150, batch_size=8, lr=0.001, device=cpu
epoch   0 | loss=0.6222 | train_dice=0.6178 | test_dice=0.6169
epoch  10 | loss=0.1979 | train_dice=0.9416 | test_dice=0.9439
epoch  20 | loss=0.1454 | train_dice=0.9525 | test_dice=0.9674
epoch  30 | loss=0.1365 | train_dice=0.9535 | test_dice=0.9680
epoch  40 | loss=0.1321 | train_dice=0.9616 | test_dice=0.9686
epoch  50 | loss=0.1270 | train_dice=0.9618 | test_dice=0.9685
epoch  60 | loss=0.1242 | train_dice=0.9619 | test_dice=0.9686
epoch  70 | loss=0.1223 | train_dice=0.9620 | test_dice=0.9686
epoch  80 | loss=0.1219 | train_dice=0.9620 | test_dice=0.9678
epoch  90 | loss=0.1217 | train_dice=0.9613 | test_dice=0.9494
epoch 100 | loss=0.1208 | train_dice=0.9616 | test_dice=0.9495
epoch 110 | loss=0.1203 | train_dice=0.9615 | test_dice=0.9495
epoch 120 | loss=0.1216 | train_dice=0.9615 | test_dice=0.9492
epoch 130 | loss=0.1202 | train_dice=0.9617 | test_dice=0.9490
epoch 140 | loss=0.1199 | tra